In [ ]:
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Dry Bean 데이터셋 도메인
 
## 수집 목적
 
시장 상황을 고려해 형태, 모양, 구조 등의 특징을 기반으로 **7종의 건조 콩 품종을 자동으로 분류**하기 위한 컴퓨터 비전 시스템을 개발하기 위해 수집됐습니다.
 
사람이 눈으로 콩을 일일이 분류하던 작업을 **머신러닝으로 자동화**하려는 농업 분야 연구입니다.
 
---
 
## 데이터 수집 방법
 
고해상도 카메라로 **13,611개의 콩 낱알**을 촬영한 뒤, 이미지 분할(segmentation)과 특징 추출(feature extraction) 과정을 거쳐 **12개의 치수(dimension) + 4개의 형태(shape) = 총 16개의 피처**를 추출했습니다.
 
> 16개 피처는 콩 이미지에서 뽑아낸 수치 (면적, 둘레, 종횡비 등)
 
---
 
## 7종 콩 클래스
 
| 클래스 | 샘플 수 | 특징 |
|--------|--------|------|
| DERMASON | 3,546 | 가장 많은 샘플, 작고 둥근 편 |
| SIRA | 2,636 | 두 번째로 많음 |
| SEKER | 2,027 | 밝고 둥근 형태 |
| HOROZ | 1,928 | 길쭉한 형태 |
| CALI | 1,630 | 큰 편 |
| BARBUNYA | 1,322 | 얼룩무늬 |
| BOMBAY | 522 | 가장 적은 샘플, 매우 큰 콩 |
 
---

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset (데이터셋 가져오기)
dry_bean = fetch_ucirepo(id=602) 
  
# data (as pandas dataframes) 
X = dry_bean.data.features 
y = dry_bean.data.targets 
  
# metadata 
print(dry_bean.metadata) 
  
# variable information (변수 정보)
print(dry_bean.variables)

{'uci_id': 602, 'name': 'Dry Bean', 'repository_url': 'https://archive.ics.uci.edu/dataset/602/dry+bean+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/602/data.csv', 'abstract': 'Images of 13,611 grains of 7 different registered dry beans were taken with a high-resolution camera. A total of 16 features; 12 dimensions and 4 shape forms, were obtained from the grains.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 13611, 'num_features': 16, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Thu Mar 28 2024', 'dataset_doi': '10.24432/C50S4B', 'creators': [], 'intro_paper': {'ID': 244, 'type': 'NATIVE', 'title': 'Multiclass classification of dry beans using computer vision and machine learning techniques', 'authors': 'M. Koklu, Ilker Ali Özkan', 'venue': 'Co

In [3]:
# 데이터 기본 확인
print(X.shape)  # 행, 열 확인
print(y.shape)  # -
print(X.isnull().sum().sum())   # 결측치 확인
print(y.value_counts()) # 클래스 분포 확인

# 13,611행 × 16피처
# 결측치 0개
# 클래스 7개 -> softmax 7
# 클래스 불균형 약간 있음 (BOMBAY 522 vs DERMASON 3546)

(13611, 16)
(13611, 1)
0
Class   
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# LabelEncoder로 y를 숫자로 변환 (DERMASON→0, SIRA→1 ...)
label = LabelEncoder()
y_label = label.fit_transform(y)

# train_test_split으로 학습/검증 분리
train_input, test_input, train_target, test_target = train_test_split(
    X, y_label, test_size=0.2, random_state=42
)

# StandardScaler로 X 스케일링 (fit은 train에만, transform은 train+val 둘 다)
scale = StandardScaler()
train_scaled = scale.fit_transform(train_input)
test_scaled = scale.transform(test_input)

# StandardScaler를 train에만 fit하는 이유
# - 테스트 데이터의 정보가 학습에 새어들어가는 걸(데이터 누수(Data Leakage)를) 막기 위해서 

c:\Users\금정산2-PC03\Desktop\git-practice\c3-deep-learning\dropout-mission\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
# DL 모델 작성
# - 16피처
# - activation = 'softmax' + 출력층 뉴런 수 = 클래스 수 (7개)
# - loss = sparse_categorical_crossentropy
# - EarlyStopping 추가 (val_loss 기준)
# - validation_data로 test_scaled, test_target 사용
model = keras.Sequential()
model.add(keras.layers.Input(shape=(16,)))
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(7, activation='softmax'))

early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss',
                                               patience=2,
                                               restore_best_weights=True) 

model.compile(optimizer='adam',
            loss=keras.losses.sparse_categorical_crossentropy,
            metrics=[keras.metrics.sparse_categorical_accuracy])
history = model.fit(train_scaled, train_target,
                    epochs=20, verbose=1,
                    validation_data=(test_scaled, test_target))

Epoch 1/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6344 - sparse_categorical_accuracy: 0.8174 - val_loss: 0.2814 - val_sparse_categorical_accuracy: 0.9137
Epoch 2/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2475 - sparse_categorical_accuracy: 0.9161 - val_loss: 0.2261 - val_sparse_categorical_accuracy: 0.9181
Epoch 3/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2178 - sparse_categorical_accuracy: 0.9222 - val_loss: 0.2092 - val_sparse_categorical_accuracy: 0.9243
Epoch 4/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2082 - sparse_categorical_accuracy: 0.9246 - val_loss: 0.2028 - val_sparse_categorical_accuracy: 0.9280
Epoch 5/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2030 - sparse_categorical_accuracy: 0.9247 - val_loss: 0.2026 - val_sparse_categorical_accuracy: 0.9262
Epoch 6/20
341/341 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1991 - sparse_categorical_accuracy: 0.9256 - val_loss: 0.1996 - val_sparse_categorical_accuracy: 0.9295
Epoc

In [8]:
# DL 최종 성능
dl_acc = history.history['val_sparse_categorical_accuracy'][-1]
print(f"DL val_accuracy: {dl_acc:.4f}")

DL val_accuracy: 0.9310


---

In [ ]:
# RF 모델 작성
# - 전처리된 train_scaled, test_scaled 그대로 사용
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(train_scaled, train_target)

# rf.predict(test_scaled)
train_acc = rf.score(train_scaled, train_target)
test_acc = rf.score(test_scaled, test_target)
print(f"RF train_accuracy: {train_acc:.4f}")
print(f"RF test_accuracy: {test_acc:.4f}")

RF train_accuracy: 1.0000
RF test_accuracy: 0.9243


## 원논문 성능 기준
 
| 모델 | 정확도 |
|------|--------|
| SVM (원논문, Koklu & Ozkan) | 93.13% |
| DL (실험) | 93.10% |
| RF (실험) | 92.43% |
 
> 원논문과 거의 동일한 수준의 성능 달성

#### "표형 데이터에서 DL이 항상 RF보다 좋은가?"

- 이번 실험 결과만 보면 DL(0.9310) > RF(0.9243)이지만, 항상 그렇다고는 할 수 없음
    - RF는 하이퍼파라미터 튜닝 없이 기본값만 써서 과적합이 심했음 -> n_estimators, max_depth 등을 조정하면 RF 성능이 올라갈 수 있음
    - 표형 데이터에서는 일반적으로 RF 계열(XGBoost 등)이 DL보다 강한 경우가 많음